I asked chatGPT if it can come up with a baseline for the Titanic competition. Easy!

<img src="https://habrastorage.org/webt/fw/ky/4x/fwky4xndgagsclzbmnu1gruqjs4.png" />

The code in the cell below is almost solely produced by chatGPT, I only fixed the path to data and created an inverse mapping from yes/no to 1/0 predictions. 

In [ ]:
import pandas as pd

# load the training and test datasets
train_df = pd.read_csv("../input/titanic/train.csv")
test_df = pd.read_csv("../input/titanic/test.csv")

# fill in missing values with the median for each column
train_df = train_df.fillna(train_df.median())
test_df = test_df.fillna(test_df.median())

# create a new column with a binary value indicating if the passenger survived or not
train_df["Survived"] = train_df["Survived"].map({0: "no", 1: "yes"})

# create a new column with a binary value indicating if the passenger is a child or not
train_df["IsChild"] = train_df["Age"].map(lambda x: "yes" if x < 16 else "no")
test_df["IsChild"] = test_df["Age"].map(lambda x: "yes" if x < 16 else "no")

# select the columns we want to use for prediction
predictors = ["Pclass", "Sex", "IsChild"]

# create dummy variables for the categorical columns
train_df = pd.get_dummies(train_df, columns=predictors)
test_df = pd.get_dummies(test_df, columns=predictors)

# select the columns we want to use for prediction
predictors = ["Pclass_1", "Pclass_2", "Pclass_3", "Sex_female", "Sex_male", "IsChild_no", "IsChild_yes"]

# initialize the algorithm and fit the data
from sklearn.linear_model import LogisticRegression
alg = LogisticRegression()
alg.fit(train_df[predictors], train_df["Survived"])

# make predictions using the test dataset
predictions = alg.predict(test_df[predictors])

# create a new column with the predictions
test_df["Survived"] = predictions
test_df["Survived"] = test_df["Survived"].map({"no": 0, "yes": 1})

# select the columns we want to submit
submission = test_df[["PassengerId", "Survived"]]

# write the submission file
submission.to_csv("submission.csv", index=False)
